In [1]:
from frameworks.LightenDiffusion.models import CTDN
from dataset_registery.registery import DatasetRegistry
from torchsummary import summary
from eda.helpers.training_helpers import train_model, get_optimizer
from eda.helpers.losses import ctdn_loss_wrapper
import torch
%load_ext autoreload
%autoreload 2

/home/grads/o/omarkhater/projects/lle-generative-priors/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "NTIRE2025"
dataset_id = "okhater/NTIRE_LLE_2025"
source = "huggingface"

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used : {device}")

Device used : cuda


In [4]:
registry = DatasetRegistry()
registry.register_dataset(
    name=data_name,
    dataset_id=dataset_id,
    splits=["train", "validation", "test"],
    dataset_type="paired",
    data_dir="../../datasets/NTIRE2025",
)


train_loader = registry.get_dataloader(data_name, "train", batch_size=32, shuffle=True)
val_loader = registry.get_dataloader(data_name, "validation", batch_size=32, shuffle=True)
test_loader = registry.get_dataloader(data_name, "test", batch_size=32, shuffle=True)

In [5]:
model = CTDN()
model = model.to(device)

In [6]:
summary(model, (3, 256, 256), batch_size=1)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [1, 64, 256, 256]           4,864
            Conv2d-2          [1, 64, 256, 256]         102,464
            Conv2d-3          [1, 64, 256, 256]          36,928
         LeakyReLU-4          [1, 64, 256, 256]               0
            Conv2d-5          [1, 64, 256, 256]          36,928
            Conv2d-6          [1, 64, 256, 256]           4,160
         Res_block-7          [1, 64, 256, 256]               0
            Conv2d-8          [1, 64, 128, 128]          36,928
            Conv2d-9         [1, 128, 128, 128]          73,856
        LeakyReLU-10         [1, 128, 128, 128]               0
           Conv2d-11         [1, 128, 128, 128]         147,584
           Conv2d-12         [1, 128, 128, 128]           8,320
        Res_block-13         [1, 128, 128, 128]               0
           Conv2d-14           [1, 128,

In [7]:
optimizer=get_optimizer(model)


In [8]:
model, losses = train_model(
        model,
        data_loaders=(train_loader, val_loader),
        criterion=ctdn_loss_wrapper, 
        optimizer=optimizer, 
        num_epochs = 100, 
        batch_size= 32, 
        device=device, 
        val_frequency = 5,
        patience = 5
        )

Epoch 1/100: 100%|██████████| 7/7 [01:22<00:00, 11.77s/batch, loss=0.21] 


Epoch: 0, avg_val_loss = 0.34793422


Epoch 6/100: 100%|██████████| 7/7 [01:22<00:00, 11.76s/batch, loss=0.159]


Epoch: 5, avg_val_loss = 0.27187991


Epoch 11/100: 100%|██████████| 7/7 [01:23<00:00, 11.90s/batch, loss=0.138]


...

Epoch: 20, avg_val_loss = 0.32004401


Epoch 26/100: 100%|██████████| 7/7 [01:22<00:00, 11.83s/batch, loss=0.196]


Epoch: 25, avg_val_loss = 0.36096856


Epoch 31/100: 100%|██████████| 7/7 [01:20<00:00, 11.45s/batch, loss=0.217]


Epoch: 30, avg_val_loss = 0.31325290
Early stopping triggered at epoch 31
Best epoch: 6, Best loss: 0.2718799114227295
